# Deliverable A: Senior Data Engineer Cleaning & Integration Pipeline
**Qiyas Data Science & AI Hackathon - Addis Ride Demand Forecasting Challenge**

This notebook implements the complete 20-step Senior Data Engineer cleaning architecture:
1. **Canonical Zone Dimension (`dim_zone.csv`)**
2. **Timezone-Smart Datetime Parsing (`Africa/Addis_Ababa` UTC+3)**
3. **Weather De-duplication (Median Aggregation) & Missingness Indicators**
4. **Event Multi-Zone Expansion & Intermediate `event_zone_hour.csv` Table**
5. **Validated Left Merges (`many_to_one` for weather, `one_to_one` for events)**
6. **Mandatory Integrity Asserts (`assert master_train.duplicated(['zone_id', 'pickup_datetime']).sum() == 0`)**

In [ ]:
# Setup & Imports
import os, sys, re
import numpy as np
import pandas as pd

base_dir = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
raw_dir_1 = os.path.join(base_dir, 'dataset', 'raw')
raw_dir_2 = os.path.join(base_dir, 'data', 'raw')
raw_dir = raw_dir_1 if os.path.exists(raw_dir_1) else raw_dir_2

processed_dir = os.path.join(base_dir, 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)

print('Environment and path setup complete.')

## 1. Canonical Zone Dimension & Mapping Functions

In [ ]:
CANONICAL_ZONES = [
    'Bole', 'Kazanchis', 'Piazza', 'Merkato', 'Sarbet', 'CMC',
    'Megenagna', 'Gerji', 'Jemo', 'Lebu', 'Gotera', 'Ayat'
]

ZONE_MAP = {
    'bole': 'Bole', 'bole rd': 'Bole', 'bole airport': 'Bole',
    'kazanchis': 'Kazanchis', 'kazanchis (kirkos)': 'Kazanchis', 'kazanchis bus': 'Kazanchis',
    'piassa': 'Piazza', 'piazza': 'Piazza', 'piazza central': 'Piazza',
    'mercato': 'Merkato', 'merkato': 'Merkato',
    'sarbet': 'Sarbet', 'cmc': 'CMC', 'megenagna': 'Megenagna',
    'gerji': 'Gerji', 'jemo': 'Jemo', 'lebu': 'Lebu', 'gotera': 'Gotera',
    'ayat': 'Ayat', 'ayat zone': 'Ayat'
}

def build_dim_zone():
    return pd.DataFrame({'zone_id': range(1, len(CANONICAL_ZONES) + 1), 'zone_name': CANONICAL_ZONES})

dim_zone = build_dim_zone()
display(dim_zone)

## 2. Timezone-Smart Datetime Parsing (UTC+3 Addis Ababa Time)
Naive timestamps interpreted directly as Addis local time. Explicit UTC `Z` strings parsed as UTC then converted to UTC+3.

In [ ]:
def parse_addis_datetime(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    has_tz = s.str.contains('Z|\\+|\\-', regex=True)
    parsed = pd.Series(index=series.index, dtype='datetime64[ns]')
    if has_tz.any():
        tz_part = pd.to_datetime(s[has_tz], format='mixed', errors='coerce', utc=True)
        parsed[has_tz] = tz_part.dt.tz_convert('Africa/Addis_Ababa').dt.tz_localize(None)
    if (~has_tz).any():
        naive_part = pd.to_datetime(s[~has_tz], format='mixed', errors='coerce')
        parsed[~has_tz] = naive_part
    return parsed.dt.floor('h')

print('Timezone-smart datetime parser defined successfully.')

## 3. Weather Cleaning & De-duplication (Median Aggregation)

In [ ]:
def clean_weather(raw_weather: pd.DataFrame) -> pd.DataFrame:
    df = raw_weather.copy()
    df['weather_datetime'] = parse_addis_datetime(df['timestamp'])
    df = df[df['weather_datetime'].notna()].copy()
    num_cols = ['temp_c', 'rain_mm', 'humidity_pct', 'wind_kmh']
    for col in num_cols:
        if col in df.columns:
            df[col] = df[col].replace([-9999, -999, -99], np.nan)
    agg_dict = {col: 'median' for col in num_cols if col in df.columns}
    if 'data_type' in df.columns:
        agg_dict['data_type'] = 'first'
    weather_clean = df.groupby('weather_datetime', as_index=False).agg(agg_dict)
    for col in num_cols:
        if col in weather_clean.columns:
            weather_clean[f'{col}_missing'] = weather_clean[col].isna().astype(int)
            weather_clean[col] = weather_clean[col].fillna(weather_clean[col].median())
    return weather_clean.sort_values('weather_datetime').reset_index(drop=True)

raw_weather = pd.read_csv(os.path.join(raw_dir, 'weather_hourly.csv'))
weather_clean = clean_weather(raw_weather)
print(f'Cleaned weather rows: {len(weather_clean)}')
display(weather_clean.head())

## 4. Multi-Zone Event Expansion & Intermediate `event_zone_hour` Feature Table

In [ ]:
def map_zone_to_id(zone_series: pd.Series, dim_zone: pd.DataFrame) -> tuple:
    name_to_id = dict(zip(dim_zone['zone_name'], dim_zone['zone_id']))
    mapped_ids, mapped_names = [], []
    for raw_val in zone_series:
        if pd.isna(raw_val):
            mapped_ids.append(np.nan)
            mapped_names.append(np.nan)
            continue
        s = re.sub(r'\s+', ' ', str(raw_val).strip().lower())
        if s in ['citywide', 'city-wide', 'all', 'all zones']:
            mapped_ids.append(0)
            mapped_names.append('Citywide')
            continue
        canonical = ZONE_MAP.get(s, str(raw_val).strip().title())
        if canonical in name_to_id:
            mapped_ids.append(name_to_id[canonical])
            mapped_names.append(canonical)
        else:
            mapped_ids.append(np.nan)
            mapped_names.append(canonical)
    return pd.Series(mapped_ids, index=zone_series.index), pd.Series(mapped_names, index=zone_series.index)

raw_events = pd.read_csv(os.path.join(raw_dir, 'events_calendar.csv'))
print('Zone mapping helper defined.')

## 5. Build Master Datasets & Execute Integrity Asserts

In [ ]:
from src.cleaning_pipeline import execute_pipeline
execute_pipeline()
print('Master datasets built and mandatory assertions passed!')